# Conformal Backfill Retrieval — Section 2.3 GPU runner

This notebook downloads the **official labelled train+development data and dataset-provided candidate pools** for MMQA, WebQA, HotpotQA, and TAT-QA; generates frozen modality-aware cosine top-$L$ retrieval logs; builds the false-match reference banks; and downloads one result ZIP. No manual file upload is required and no synthetic WebQA distractors are created.

This step uses **Jina CLIP v2 embeddings, not Qwen generation**, so vLLM is neither required nor useful. An L4 24 GB GPU should run it; an A100 is faster. CUDA OOM during encoding automatically halves the affected batch and retries.

> `MAX_QUESTIONS_PER_DATASET = 0` means all publicly labelled train+dev questions. Official WebQA images require a one-time ~39 GB archive download, so disk/network can dominate GPU time. Use a positive limit for an L24 timing trial; the candidates and labels remain official.

In [ ]:
# Colab setup — all Python packages are installed with uv.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/NguyenKhanh2603/Uncertainty-Aware-Iterative-RAG.git"
CODE_COMMIT = "a25cf3a1faf0e39b66250080ad3a7981873335a0"
PROJECT_DIR = Path("/content/uncertainty-aware-rag")

uv_executable = shutil.which("uv")
if uv_executable is None:
    installer = Path("/tmp/install-uv.sh")
    subprocess.check_call(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", str(installer)])
    subprocess.check_call(["sh", str(installer)])
    uv_executable = str(Path.home() / ".local" / "bin" / "uv")
if not Path(uv_executable).is_file():
    raise RuntimeError(f"uv installation failed: {uv_executable}")

packages = [
    "transformers==4.46.3",
    "huggingface_hub>=0.26.0",
    "pyarrow>=16.0.0",
    "remotezip>=0.12.3",
    "requests>=2.31.0",
    "einops>=0.8.0",
    "timm>=1.0.0",
    "Pillow>=10.0.0",
    "numpy>=1.26.0",
    "tqdm>=4.66.0",
    "safetensors>=0.4.0",
]
subprocess.check_call([uv_executable, "pip", "install", "--system", *packages])

if (PROJECT_DIR / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "fetch", "origin", CODE_COMMIT])
elif PROJECT_DIR.exists():
    raise RuntimeError(f"{PROJECT_DIR} exists but is not a git checkout; remove or rename it")
else:
    subprocess.check_call(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(PROJECT_DIR)])
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "fetch", "origin", CODE_COMMIT])
subprocess.check_call(["git", "-C", str(PROJECT_DIR), "checkout", "--detach", CODE_COMMIT])

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU, then rerun this cell")
gpu = torch.cuda.get_device_properties(0)
import transformers
import torchvision
print(f"Ready: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB)")
print(f"Python={sys.version.split()[0]} torch={torch.__version__} torchvision={torchvision.__version__} transformers={transformers.__version__}")
print(f"Code: {REPO_URL} commit={CODE_COMMIT}")

## Configuration

The default uses every publicly labelled train+dev question. Only the official training split is divided into development/calibration; official dev/validation is reserved exclusively as final test, and official test files are excluded. For a quick hardware trial, set `MAX_QUESTIONS_PER_DATASET = 1000`; this limits questions but does not synthesize candidates. Each query is ranked only against the candidate pool supplied by that benchmark. Progress bars report elapsed time and ETA for preparation, corpus/query encoding, candidate ranking, and log writing.

In [ ]:
DATASETS = ["mmqa", "webqa", "hotpotqa", "tatqa"]
MAX_QUESTIONS_PER_DATASET = 0  # 0 = ALL labelled train+dev; use 1000 for an L24 timing trial
DATA_GRADE = "paper" if MAX_QUESTIONS_PER_DATASET == 0 else "smoke"
TOP_L = 30
RETRIEVAL_MODE = "modality_aware"
MIN_PER_MODALITY = 10
BANK_CONDITIONING = "dataset,modality"
RANK_BINS = "1-3,4-10,11-30"  # only used if rank_bin is added to BANK_CONDITIONING
MIN_BANK_SIZE = 1000
QUERY_TYPE_MODE = "pooled"
SPLIT_POLICY = "official_holdout"  # train -> dev/calibration; official dev/validation -> test only
TEXT_BATCH_SIZE = 32       # reduce manually only if repeated batch-size-1 OOM occurs
IMAGE_BATCH_SIZE = 8
QUERY_BATCH_SIZE = 64
CORPUS_BLOCK_SIZE = 16384
SEARCH_CORPUS_ON_GPU = "auto"  # safe on L24; falls back to CPU-resident corpus if needed
OFFICIAL_BUNDLE_DIR = Path("/content/conformal_official_bundle")
OUTPUT_DIR = Path("/content/conformal_backfill_2_3_results")
CACHE_DIR = Path("/content/conformal_backfill_embedding_cache")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Question limit per dataset:", "ALL" if MAX_QUESTIONS_PER_DATASET == 0 else MAX_QUESTIONS_PER_DATASET)
print("Important: full WebQA needs a one-time ~39 GB image archive download.")

## Prepare official benchmark data

This writes a normalized corpus plus per-query official candidate/support IDs. Downloads resume through their source caches. WebQA performs a disk preflight before starting its large image download.

In [ ]:
import select
import time

def run_streamed(command, *, cwd, env, log_path, heartbeat_seconds=15):
    """Stream a child process to Colab and emit a heartbeat while it is silent."""
    child_env = env.copy()
    child_env["PYTHONUNBUFFERED"] = "1"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(map(str, command)), flush=True)
    started = time.monotonic()
    last_heartbeat = started
    with log_path.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command, cwd=cwd, env=child_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=False, bufsize=0,
        )
        assert process.stdout is not None
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 5.0)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if chunk:
                    text_chunk = chunk.decode("utf-8", errors="replace")
                    print(text_chunk, end="", flush=True)
                    log_handle.write(text_chunk)
                    log_handle.flush()
            now = time.monotonic()
            if process.poll() is not None:
                remainder = process.stdout.read()
                if remainder:
                    text_chunk = remainder.decode("utf-8", errors="replace")
                    print(text_chunk, end="", flush=True)
                    log_handle.write(text_chunk)
                    log_handle.flush()
                break
            if now - last_heartbeat >= heartbeat_seconds:
                message = f"[still running] elapsed={(now - started) / 60:.1f} min | log={log_path}\n"
                print(message, end="", flush=True)
                log_handle.write(message)
                log_handle.flush()
                last_heartbeat = now
    return process.returncode

prepare_script = PROJECT_DIR / "scripts" / "prepare_official_conformal_bundle.py"
prepare_command = [
    sys.executable, str(prepare_script),
    "--output-dir", str(OFFICIAL_BUNDLE_DIR),
    "--datasets", ",".join(DATASETS),
    "--max-questions", str(MAX_QUESTIONS_PER_DATASET),
]
prepare_log = Path("/content/official_bundle_prepare.log")
return_code = run_streamed(
    prepare_command, cwd=PROJECT_DIR, env=os.environ, log_path=prepare_log
)
if return_code != 0:
    tail = prepare_log.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]
    raise RuntimeError(
        f"Official-data preparation failed with exit code {return_code}. "
        f"Full output is above and in {prepare_log}.\n" + "\n".join(tail)
    )
import json
official_manifest = json.loads((OFFICIAL_BUNDLE_DIR / "manifest.json").read_text())
print("Official questions:", {name: info["questions"] for name, info in official_manifest["datasets"].items()})
print(f"Preparation time: {official_manifest['total_seconds'] / 60:.1f} min")

## Generate frozen top-L retrieval logs on GPU

Each dataset gets a JSONL.GZ log and a manifest. Corpus embeddings are cached, so rerunning a completed dataset avoids recomputing them. The final `[PERFORMANCE]` line separates encoding and search time and records the actual GPU/VRAM.

In [ ]:
runner = PROJECT_DIR / "scripts" / "generate_conformal_retrieval_log.py"
python_env = os.environ.copy()
python_env["PYTHONPATH"] = str(PROJECT_DIR / "src") + os.pathsep + python_env.get("PYTHONPATH", "")
python_env["PYTHONUNBUFFERED"] = "1"
log_paths = []
for dataset in DATASETS:
    output_path = OUTPUT_DIR / f"{dataset}_top{TOP_L}_retrieval.jsonl.gz"
    command = [
        sys.executable, str(runner),
        "--dataset", dataset,
        "--bundle-dir", str(OFFICIAL_BUNDLE_DIR),
        "--output", str(output_path),
        "--cache-dir", str(CACHE_DIR),
        "--device", "cuda",
        "--dtype", "float16",
        "--top-l", str(TOP_L),
        "--retrieval-mode", RETRIEVAL_MODE,
        "--min-per-modality", str(MIN_PER_MODALITY),
        "--query-type-mode", QUERY_TYPE_MODE,
        "--split-policy", SPLIT_POLICY,
        "--text-batch-size", str(TEXT_BATCH_SIZE),
        "--image-batch-size", str(IMAGE_BATCH_SIZE),
        "--query-batch-size", str(QUERY_BATCH_SIZE),
        "--corpus-block-size", str(CORPUS_BLOCK_SIZE),
        "--search-corpus-on-gpu", SEARCH_CORPUS_ON_GPU,
        "--non-support-label", "false",
        "--data-grade", DATA_GRADE,
    ]
    print(f"\n===== {dataset} =====", flush=True)
    process_log = OUTPUT_DIR / f"{dataset}_runner.log"
    return_code = run_streamed(
        command, cwd=PROJECT_DIR, env=python_env, log_path=process_log
    )
    if return_code != 0:
        tail = process_log.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]
        raise RuntimeError(
            f"{dataset} retrieval failed with exit code {return_code}. "
            f"Full child-process output is above and in {process_log}.\n"
            + "\n".join(tail)
        )
    log_paths.append(output_path)
print("Finished retrieval logs:", [path.name for path in log_paths])

## Build the combined false-match reference banks

Banks use calibration queries only. The 1,000-false-score gate remains active; `--allow-small-banks` lets a limited timing trial finish while clearly listing underpowered banks.

In [ ]:
bank_builder = PROJECT_DIR / "scripts" / "prepare_conformal_reference_banks.py"
bank_path = OUTPUT_DIR / "combined_reference_banks.json.gz"
subprocess.check_call([
    sys.executable, str(bank_builder),
    "--input", *[str(path) for path in log_paths],
    "--output", str(bank_path),
    "--rank-bins", RANK_BINS,
    "--conditioning", BANK_CONDITIONING,
    "--min-bank-size", str(MIN_BANK_SIZE),
    "--allow-small-banks",
], cwd=PROJECT_DIR, env=python_env)

import gzip
import json
with gzip.open(bank_path, "rt", encoding="utf-8") as handle:
    banks = json.load(handle)
print(json.dumps({
    "is_paper_ready": banks["is_paper_ready"],
    "top_l": banks["top_l"],
    **banks["summary"],
}, indent=2))
if not banks["is_paper_ready"]:
    print("Some banks are underpowered. Increase MAX_QUESTIONS_PER_DATASET or pool conditioning before a formal claim.")

timing_rows = []
for dataset in DATASETS:
    manifest = json.loads(Path(f"{OUTPUT_DIR}/{dataset}_top{TOP_L}_retrieval.jsonl.gz.manifest.json").read_text())
    timing_rows.append({"dataset": dataset, "questions": manifest["queries"], **manifest["timings_seconds"]})
print("\nMeasured runtime by stage (seconds):")
for row in timing_rows:
    print(row)
total_gpu_minutes = sum(row["total"] for row in timing_rows) / 60
print(f"Total measured retrieval runtime: {total_gpu_minutes:.1f} min (official-data preparation reported separately above).")
OFFICIAL_LABELLED_COUNTS = {"mmqa": 26258, "webqa": 41732, "hotpotqa": 97852, "tatqa": 14883}
if MAX_QUESTIONS_PER_DATASET > 0:
    projected_seconds = sum(
        row["total"] * OFFICIAL_LABELLED_COUNTS[row["dataset"]] / max(1, row["questions"])
        for row in timing_rows
    )
    print(f"Rough full-run projection from this server: {projected_seconds / 3600:.1f} hours. This is conservative/imperfect because corpus chunks are deduplicated and WebQA's 39 GB preparation cost is mostly fixed.")
if torch.cuda.get_device_properties(0).total_memory >= 22 * 2**30:
    print("VRAM verdict: an L24-class 24 GB GPU is sufficient for this embedding pipeline if SEARCH_CORPUS_ON_GPU='auto'. Choose another server only for better throughput/disk/network, not because Qwen-sized VRAM is required.")

## Package and download results

In [ ]:
archive_base = Path("/content/conformal_backfill_2_3_results")
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR))
print(f"Created {archive_path} ({archive_path.stat().st_size / 2**20:.2f} MiB)")
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("Not running in Colab; download manually from:", archive_path)